# Chapter 1 Preliminaries


**一切的深度学习问题都包括四个主要组成：1.数据 2.能够将输入转为预测的模型 3. 一个能够评估模型所作的损失 4. 一个能够更新模型参数以降低损失的算法**

### 推荐学习资源
Resources and Further Reading  The resources are grouped into books, courses, and documentation, and each names the section it accompanies. All are freely accessible online.  Books  • Python for Data Analysis, 3rd ed. — Wes McKinney — an open‐access guide to NumPy and pandas by the creator of pandas; a companion to Section 1.1 and Section 1.2. • Python Data Science Handbook — Jake VanderPlas — an example‐driven tour of NumPy, pandas, and the wider scientific‐Python stack used throughout the book. • Mathematics for Machine Learning — Deisenroth, Faisal & Ong — a development of linear algebra, calculus, and probability aimed at machine learning, in more depth than Section 1.3, Section 1.4, and Section 1.6. • Introduction to Probability — Blitzstein & Hwang — a free PDF of the textbook of Harvard’s Stat 110, whose lecture videos and materials are also online; a fuller treatment of Section 1.6.  Courses and video lectures  • Essence of Linear Algebra — 3Blue1Brown — animated lectures on vectors and matrices as geometric transformations; a visual companion to Section 1.3. • Essence of Calculus — 3Blue1Brown — animated lectures on limits, derivatives, and the chain rule; a visual companion to Section 1.4 and Section 1.5. • NumPy: the absolute basics for beginners — NumPy project — the official introduction to the array operations of Section 1.1. • NumPy Tutorials — NumPy project — runnable notebooks that apply array programming to concrete data tasks.  Tutorials, notes, and documentation  • Array programming with NumPy — Harris, Millman, van der Walt et al. (Nature, 2020) — the reference paper on the n‐dimensional array model that every tensor library in this book follows (preprint at arXiv:2006.10256). • NumPy user guide — NumPy project — the official reference for array creation, indexing, broadcasting, and data types (Section 1.1). • pandas user guide — pandas project — the official reference for the loading, indexing, and missing‐data handling of Section 1.2. • PyTorch documentation — PyTorch project — the official API reference for tensor operations and automatic differentiation; Section 1.7 lists the corresponding references for JAX, TensorFlow, and MXNet and shows how to use them. • SciPy documentation — SciPy project — routines for lin

## 1.1 Data Manipulation

*minibatches*：训练一个模型的数据集的一个小样本集合--小样本集
*tensor(张量)*：任意轴组织的数字叫做张量

In [34]:
import torch

### 1.1.1 Tensor Creation

#### 1.1.1.1 Shape

In [35]:
x = torch.arange(12,dtype=torch.float32) # 生成一个有序张量，每个元素是32位浮点数表示
print(x)

tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11.])


可以看到上述生成的张量是由浮点数保存一个从0-11的有序张量。

**元素**：张量中的每个数字被称作元素
**大小**：这里的大小一般指的是一个张量的大小，即一个张量的数字的个数。PyTorch称大小为numel，使用.numel可以查看一个张量的大小：

In [36]:
x.numel()

12

**Shape(形状)**：张量的各个轴的长度，比如二维张量各个轴方向可以指定形状，比如：


In [37]:
torch.randn(3,4)    #这里指定的是张量的x轴长度为3，y轴长度为4.


tensor([[-0.1551,  2.0444, -2.1396, -0.8615],
        [ 0.8092, -0.8210,  1.6214, -0.2428],
        [-1.8541, -0.8379, -2.4239, -0.1621]])

这里生成了一个随机张量，这里说明一下常见的维度导致不同的张量类型，0维张量称作**标量**，1维张量称作**向量**，二维张量称作**矩阵**，更高维暂时没有不定的说法，但是能够理解高维度的张量即可。

#### 1.1.1.3 Data Types

**Data Type(数据类型)**:每一个张量都有对应的数据类型，也就是使用多大的**精度/内存**来存储张量内部数据。
*模型越大越有可能需要高精度，但也不一定，其次比如QLora微调方式就是通过降精度来减少内存消耗，但是效果同样能够达到全精度微调的90%+，所以是常见的有限资源下的微调方式之一*

#### 1.1.1.3 Initial Values
模型需要向量已经被初始化之后才根据训练更新参数达到训练模型的目的最简单的方式就是直接使用zeros|ones创建一个全0或者全1的张量，不过**从零到一的模型训练推荐随机初始化参数**，比如全0参数无法更新等原因，后面第四章会细讲。

In [38]:
torch.zeros((2,3,4)) #三维全零张量

tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

以下是随机张量函数的展示

In [39]:
torch.randn(3,4)    #已经解释过了，不过多阐述。
torch.random.initial_seed() #每次随机生成张量是需要生成种子的，如果没有指定，那么torch会采用懒加载，也就是在真正调用randn函数时才生成种子用于生成张量

42

当然，随机也不是完全随机的，你可以设置当前pytorch的随机生成器的生成种子（正常情况下也是系统随机生成的），不过也可以指定，**相同的种子生成的相同维度的张量是相同的：

In [40]:
torch.manual_seed(42)
a = torch.randn(3) 
b = torch.randn(3) # The generator has moved on ，理解为种子已经+1了
torch.manual_seed(42) 
c = torch.randn(3) # Same seed: c repeats a 
torch.manual_seed(42) 
d,e = torch.randn(4,5),torch.randn(4)
a, b, c,d,e

(tensor([0.3367, 0.1288, 0.2345]),
 tensor([ 0.2303, -1.1229, -0.1863]),
 tensor([0.3367, 0.1288, 0.2345]),
 tensor([[ 1.9269,  1.4873,  0.9007, -2.1055, -0.7581],
         [ 1.0783,  0.8008,  1.6806,  0.3559, -0.6866],
         [-0.4934,  0.2415, -0.2316,  0.0418, -0.2516],
         [ 0.8599, -0.3097, -0.3957,  0.8034, -0.6216]]),
 tensor([-0.7658, -0.7506,  1.3525,  0.6863]))

*注意：这里的e&a不一样，但是c&a确实一样的原因是每生成一个元素都需要一个随机数，也就是在生成d的时候已经消耗了指定种子后续的4\*5=20个随机数流，那么生成e的时候是由42种子产生的第21个随机数，自然后续生成的张量也不是一样的，如果要一样需要重新指定种子*

最后一个方式就是直接手动指定张量的元素，比如：

In [41]:
torch.tensor([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

tensor([[2, 1, 4, 3],
        [1, 2, 3, 4],
        [4, 3, 2, 1]])

同样的，可以指定生成的范围

In [42]:
# 256张灰度图，尺寸28×28像素，每个像素是[0, 255]范围内的整数
images = torch.randint(0, 256, (256, 1, 28, 28), dtype=torch.uint8) 
images.shape, images.dtype
# 输出：(torch.Size([256, 1, 28, 28]), torch.uint8)

(torch.Size([256, 1, 28, 28]), torch.uint8)

其中，torch.randit函数的前两个数字指定范围、指定形状的随机整数张量
生成的是：**一个 4 维张量，代表一批假的灰度图片**

拆开看：

- 一共 **256 张图片**（batch=256）
- 每张是**灰度图：1 通道**（如果是 RGB 就是 3 通道）
- 分辨率：28 行 × 28 列（28×28 像素）
- 每一个像素的值：**0～255 的随机整数**，类型是 uint8（图像标准类型）
- 这些不是真实衣服图片，是一堆随机噪点图（雪花图），只是用来凑出标准图像张量形状，给模型测试用。

张量维度含义（PyTorch NCHW）：
`[N, C, H, W] = [样本数, 通道数, 高, 宽]`

### 1.1.2 Reshaping and Indexing

#### 1.1.2.1 Reshaping

.reshape函数改变张量的形状，但是不能改变其大小&它的元素顺序。

In [43]:
X = x.reshape(3,4) #将一维张量转为二维张量，大小&底层存储/顺序不变。
images.reshape(256,-1).shape


torch.Size([256, 784])

其中-1参数是根据大小/顺序不变的的前提下，通过其他维度的shape自行推断当前维度的shape

同样我们可以给张量添加一个“填充维度”，尤其是如果模型希望输出是(1,1,28,28)的张量的时候，但是我们的数据是(1,28,28)的张量，那么就需要新增一个轴（维度）来创建新的张量用于临时训练，可以通过*None*&*squeeze()*做到

In [44]:
image = images[0]
image[None].shape,image.squeeze(0).shape

(torch.Size([1, 1, 28, 28]), torch.Size([28, 28]))

#### 1.1.2.2 Indexing and Slicing

**主要介绍几种张量中元素的取法：单索引、切片、张量对应、掩码、掩码替换**

这里的从张量中取值和python中取值一样，需要对张量进行**索引**，这里的张量索引&切片都是和python一样的语法：
1. 索引从0开始，到n-1结束
2. 切片[],左闭右开的形式

In [45]:
X[-1],X[1:3]

(tensor([ 8.,  9., 10., 11.]),
 tensor([[ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]))

注意：这里的整数索引只去了一行，所以看着有点像是个向量，而切片却保留了两行，但是**整数取行会去掉行的维度，而切片会保留原有的张量维度。**

In [46]:
rows,cols= torch.arange(3),torch.tensor([3,0,2])    #行&列分别匹配取张量中的元素
X[rows,cols]


tensor([ 3.,  4., 10.])

这里是高级的张量取值用法，通过多个张量一一对应匹配的方式取张量中的元素。


In [47]:
mask = (X>2)&(X<8)
X[mask]

tensor([3., 4., 5., 6., 7.])

这里通过指定**掩码**的方式实现对X张量中满足掩码要求的取值。


In [48]:
torch.where(mask,X,0)

tensor([[0., 0., 0., 3.],
        [4., 5., 6., 7.],
        [0., 0., 0., 0.]])

这里通过torch.where()函数实现对X张量的过滤，并将不符合掩码要求的替换为指定元素0

#### 1.1.2.3 Writing Elements
**TensorFlow&JAX的tensor是不可变的，一旦创建就不可更改**
*我们知道tensor中的元素在内存中一个紧挨着另外一个，也就是紧密排列。*

**View(视图)**：如果一个tensor使用的是共享的内存（也就是两个及以上的tensor底层共享的是同一个物理内存存储的元素）就叫做View。比如x&X底层就是共享的内存，那么一旦更改，同样的另外一方的元素也是被更改后的（和python的引用一样的原理）

In [49]:

X[1,2] = 17
x,X


(tensor([ 0.,  1.,  2.,  3.,  4.,  5., 17.,  7.,  8.,  9., 10., 11.]),
 tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5., 17.,  7.],
         [ 8.,  9., 10., 11.]]))

这里的将X的第二行第三列的元素修改为17，x中的元素也会变成17

同样的可以使用切片快速重写，比如

In [50]:
X[:2,:] = 12 #第一行&第二行的所有列全都写为12
X



tensor([[12., 12., 12., 12.],
        [12., 12., 12., 12.],
        [ 8.,  9., 10., 11.]])

### 1.1.3 Operations
#### 1.1.3.1 Elementwise Operations

张量满足加减乘除的基础算法，同样的是张量间一一对应。

In [51]:
x = torch.tensor([1.0, 2, 4, 8]) 
y = torch.tensor([2.0, 2, 2, 2])
x + y, x - y, x * y, x / y, x ** y

(tensor([ 3.,  4.,  6., 10.]),
 tensor([-1.,  0.,  2.,  6.]),
 tensor([ 2.,  4.,  8., 16.]),
 tensor([0.5000, 1.0000, 2.0000, 4.0000]),
 tensor([ 1.,  4., 16., 64.]))

自然指数函数：

In [52]:
torch.exp(x)

tensor([2.7183e+00, 7.3891e+00, 5.4598e+01, 2.9810e+03])

/总是返回浮点数，这里为float32.

In [53]:
scaled = images / 255
print(scaled.dtype,images[0,0,0,:4],scaled[0,0,0,:4],sep='\n')

torch.float32
tensor([219, 187, 207,  14], dtype=torch.uint8)
tensor([0.8588, 0.7333, 0.8118, 0.0549])


这里的int8的images的tensor转为了float32.

**张量间元素级别的比较：**

In [54]:
x == y

tensor([False,  True, False, False])

**取张量的总和：**

In [55]:
x.sum(),(x==y).sum()    #第二个，True=1，False=0，sum=1

(tensor(15.), tensor(1))

#### 1.1.3.2 Broadcasting(广播)
刚刚讨论的操作大多是在两个张量形状相同的前提下，那么如果不同呢？
**广播**是深度学习库从NumPy中采用的一种约定，它将向量应用于每一行，而不复制向量。

In [56]:
x= torch.arange(12,dtype=torch.float32).reshape(3,4)
v = torch.tensor([100.0,200,300,400])
x + v

tensor([[100., 201., 302., 403.],
        [104., 205., 306., 407.],
        [108., 209., 310., 411.]])

以上的结果对没一行都进行了张量相加。如果不兼容，那么会报错。
如果两个张量 x 和 y “可广播”，则结果张量的大小按以下方式计算：收听网络电台

- 如果 x 和 y 的维度数量不相等，则在维度较少的张量前面补 1，使其维度长度相等。

- 然后，对于每个维度大小，结果维度大小是 x 和 y 在该维度上的大小的最大值。
以下有更深刻的行级广播的例子。

In [57]:
a = torch.arange(3)
a


tensor([0, 1, 2])

In [58]:
a = torch.arange(3).reshape(3,1)
b = torch.arange(2).reshape(1,2)
a,b

(tensor([[0],
         [1],
         [2]]),
 tensor([[0, 1]]))

In [59]:
a+b

tensor([[0, 1],
        [1, 2],
        [2, 3]])

如果维度不匹配且无法自动扩充呢？

In [60]:
try:  
    torch.ones((3, 2)) + torch.ones((2, 3))
except Exception as e:  
    print(e)

The size of tensor a (2) must match the size of tensor b (3) at non-singleton dimension 1


广播的主要步骤有如下：
1. 对齐规则：把两个 tensor 的 shape **从最右边（最后一维）开始对齐**，短的 shape 左边自动补 1，补齐长度。
2. 校验规则：对齐之后，同一位置的两个维度大小，需要满足至少其一才合法：
    - 相等
    - 其中一个等于1(可自动扩展)
3. 自动扩充：如果校验规则符合第二条，那么才有此步，**size=1 的那一维，会被拉伸到对面维度的大小**。

⚠️ 重点：**不是复制一份份数据存进内存！**
PyTorch 修改 `stride`（步长），访问的时候反复读取同一个元素，逻辑上看起来变成多行 / 多列。

In [61]:
#1.A:(3,2),B(1,2) 维度相同，不用补齐长度，如果B(2,)，那么会触发对齐规则将B补为B(1,2)
#2. A(3,2),B(3,2) 对齐之后且各自维度大小相等，如果B(2,3)，那么将会报错，不能够成功匹配
#3. A(3,2),B(1,2) 对齐&校验之后自动扩充为'B(3,2)'

#### 1.1.3.3 Concatenation&Stacking
**拼接**是指定两个张量沿着某一个轴进行拼接达到某一个维度长度增加的目的：

In [62]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4) 
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
torch.cat((X,Y),dim=0),torch.cat((X,Y),dim=1)   #按照第1维拼接 #第二维拼接 


(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [ 2.,  1.,  4.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 4.,  3.,  2.,  1.]]),
 tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
         [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
         [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]]))

**Stacking(堆叠)**:可以理解为通过引入外部轴来定位X或者Y，然后实现‘升维’

In [63]:
torch.stack((X, Y)).shape, torch.stack((X, Y), dim=2).shape


(torch.Size([2, 3, 4]), torch.Size([3, 4, 2]))

### 1.1.4 Memory and Conersion
#### 1.1.4.1 Allocation(分配/内存分配) and In-Place Updates
大多数的操作都会重新分配内存，组成新的张量然后重新赋值给给定变量，比如：

In [64]:
before = id(Y)
Y = Y+X #X&Y相加运算之后重新赋值，已经不是原来的Y了
id(Y) == before

False

张量是会占用内存的，比如一个十亿参数的模型，需要4GB的内存存储float32精度，仅仅是存储参数，而推理&训练过程中，每一步都需要更新这些参数，所以说内存管理也是诗人重要的。
而切片是基于原对象重新赋值的，也就是原地。

In [65]:
Z = torch.zeros_like(Y)
before = id(Z) 
Z[:] = X + Y 
id(Z) == before

True

**增广**张量同样也是原地进行增广，比如以下的X还是原来的X：

In [66]:
before = id(X)
X += Y
id(X) == before


True

#### 1.1.4.2 Conversion to Other Python Objects
张量当然也可以转为其他Python数据对象，比如数组，Numpy的Array等等。但是是否是复制数值重新创建对象还是和View一样只是切换视图，那么就因为库的不同而有所不同了。

In [67]:
A = X.numpy()
B = torch.from_numpy(A)
type(A),type(B)

(numpy.ndarray, torch.Tensor)

In [68]:
A[0, 0] = 100 # Write to the NumPy array 
X[0, 0], B[0, 0]

(tensor(100.), tensor(100.))

这里的numpy&tensor中底层就是使用的相同的内存，而没有新建对象，所以更改一个也会导致另外的绑定更改。

In [ ]:
a = torch.tensor(3.5) 
a, a.item(), float(a), int(a)

参数在参与运算的时候，需要一些时间&空间，甚至转换的时候导致精度的损失，所以需要尽可能避免数据类型之间转换导致的错误。

### 1.1.5 Summary

深度学习代码将其数据、参数和中间结果以张量、具有形状的数组和数据类型进行存储。它使用整个张量而不是在其元素上使用循环来计算：一个表达式将一个小批量数据的所有200，704个像素除以255。重排(reshape)、索引(索引&切片)和串联(concatation)对元素进行重新排列、选择和连接。逐元的操作在每个位置使用一个数的函数，广播(boardcast)通过从右边和长度为1的拉伸轴排列不同形状的张量来组合它们。大多数操作分配一个新的张量；在内存重要的地方，具有可变张量的库更新它们的位置，而具有不变张量的库通过可变变量或通过它们的编译器重用内存。

In [71]:
a = torch.tensor([200],dtype=torch.uint8)
b = torch.tensor([100],dtype=torch.uint8)
a+b

tensor([44], dtype=torch.uint8)

### 1.1.6 Exercise
1. 

## 1.2 Data Preprocessing
